# Migrando uma base pública com o `i3_shift_lake`

Prova de conceito: o `i3_shift_lake` foi feito para o Redshift, mas só depende de uma função `query(sql) -> pandas.DataFrame`. Aqui usamos como fonte uma base pública real (população e PIB por país/ano, Banco Mundial) materializada num [DuckDB](https://duckdb.org/) local — que fala o mesmo dialeto SQL — e rodamos o pipeline completo: **discover → extract (paginado, particionado, incremental) → transform → validate**, sem alterar o pacote.

Dados: [github.com/datasets](https://github.com/datasets) (espelho em CSV do [World Bank Open Data](https://data.worldbank.org/), CC-BY-4.0).

In [ ]:
import sys

# raiz do repo (dois niveis acima de notebooks/), para importar worldbank_lake e i3_shift_lake
sys.path.insert(0, "../..")

import pandas as pd

from worldbank_lake.source import build_database, append_new_years, get_query_fn
from i3_shift_lake import (
    discover, load_model, extract_all, TableLoader,
    check_row_counts, check_duplicates_all,
)

## Parâmetros

`YEAR_CUTOFF` corta a carga inicial nos anos até esse valor, para depois simular a chegada de anos mais novos com `append_new_years()` e demonstrar a carga incremental de verdade (o dataset em si é uma foto estática do Banco Mundial, sem atualizações reais chegando).

In [ ]:
SCHEMA = "worldbank"
CONFIG_DIR = "./config"
CONTROL_DIR = "./control"
OUTPUT_DIR = "./dados_extraidos"
PAGE_SIZE = 2_000
NUM_BUCKETS = 8
YEAR_CUTOFF = 2015  # carga inicial so ate esse ano; o resto "chega" depois

## 1. Construir a fonte pública

Baixa os CSVs (uma vez; ficam em cache em `raw/`) e monta o DuckDB local com as tabelas `population` e `gdp`, cortadas em `YEAR_CUTOFF` — simulando uma carga inicial parcial.

In [ ]:
build_database(year_cutoff=YEAR_CUTOFF)
query = get_query_fn()

from i3_shift_lake import ping
ping(query)  # valida a conexao antes de rodar o pipeline inteiro

## 2. Discover — descobrir o modelo do schema

Igual ao pipeline do Redshift: busca tabelas, colunas, tipos e contagem de linhas via `information_schema`, e persiste em `CONFIG_DIR/worldbank.json`.

In [ ]:
model = discover(query, SCHEMA, config_dir=CONFIG_DIR)

pd.DataFrame(
    [{"table_name": t, "columns": len(cols), "row_count": model.row_counts[t]} for t, cols in model.tables.items()]
)

## 3. Extract — carga inicial (paginada e particionada)

`load_mode` é `"incremental"` por padrão; sem checkpoint prévio, isso equivale a uma carga full. Repare no relatório: com `PAGE_SIZE=2_000` e ~15-17 mil linhas por tabela, a extração acontece em várias páginas de verdade.

In [ ]:
relatorio_inicial = extract_all(
    query, model, OUTPUT_DIR, page_size=PAGE_SIZE, num_buckets=NUM_BUCKETS,
    config_dir=CONFIG_DIR, control_dir=CONTROL_DIR,
)
relatorio_inicial

## 4. Chegam dados novos — carga incremental de verdade

`append_new_years()` insere na base os anos `>= YEAR_CUTOFF + 1` (que existem no CSV original mas ficaram de fora da carga inicial), com um `date_modified` mais recente — simulando a chegada de dados novos na fonte. Rodar `extract_all` de novo, sem mudar nada, lê só o que é novo.

In [ ]:
novas_linhas = append_new_years(min_year=YEAR_CUTOFF + 1)
novas_linhas

In [ ]:
model_atualizado = discover(query, SCHEMA, config_dir=CONFIG_DIR)  # refaz a contagem total

relatorio_incremental = extract_all(
    query, model_atualizado, OUTPUT_DIR, page_size=PAGE_SIZE, num_buckets=NUM_BUCKETS,
    config_dir=CONFIG_DIR, control_dir=CONTROL_DIR,
)
relatorio_incremental  # rows_read aqui deve bater com novas_linhas, nao com o total da tabela

Rodar de novo, sem nenhum dado novo, não deve trazer nenhuma linha — prova de que o checkpoint está funcionando:

In [ ]:
extract_all(
    query, model_atualizado, OUTPUT_DIR, page_size=PAGE_SIZE, num_buckets=NUM_BUCKETS,
    config_dir=CONFIG_DIR, control_dir=CONTROL_DIR,
)

## 5. Transform — trabalhar os dados em pandas

`TableLoader` lê o acumulado das duas cargas (inicial + incremental) como um único `DataFrame`.

In [ ]:
loader = TableLoader(OUTPUT_DIR, schema=SCHEMA)
df_pop = loader["population"]
df_gdp = loader["gdp"]

print(df_pop.shape, df_gdp.shape)
df_pop.sort_values(["country_code", "year"]).head()

In [ ]:
# populacao do Brasil ao longo do tempo, ja incluindo os anos que "chegaram" depois
df_pop[df_pop["country_code"] == "BRA"].sort_values("year")[["year", "value"]]

## 6. Validate — checar a carga

Linhas carregadas x esperadas (discover) e duplicidade de `id` nos parquets acumulados.

In [ ]:
check_row_counts(model_atualizado, OUTPUT_DIR)

In [ ]:
check_duplicates_all(model_atualizado, OUTPUT_DIR, config_dir=CONFIG_DIR)

## Notas

- **Nenhuma linha do `i3_shift_lake` foi alterada** para isso funcionar — só a função `query()` mudou (DuckDB em vez de Redshift). Qualquer fonte que fale SQL parecido (Postgres, MySQL com ajustes, etc.) serve do mesmo jeito.
- **`append_new_years` é só para a demo**: no mundo real, os dados novos chegam sozinhos na fonte (Redshift); aqui simulamos isso porque o dataset do Banco Mundial é uma foto estática.
- **`YEAR_CUTOFF`**: baixe para um valor mais recente (ex.: 2022) para simular um incremento pequeno, ou remova o corte (`build_database()` sem `year_cutoff`) para já carregar tudo de uma vez.
- Para recomeçar do zero (ex.: depois de mexer no `YEAR_CUTOFF`), apague `warehouse.duckdb`, `config/`, `control/` e `dados_extraidos/` antes de rodar de novo — ou use `load_mode="full"` no `extract_all` para reconstruir uma tabela específica sem apagar tudo.